# TX (Mirror) Scan Analysis — Class-based API

This notebook demonstrates the new `StepData` / `ScanData` / `ScanSet` API
for analysing CARCARÁ-X mirror-scan HDF5 data.

**Before starting**, set `workdir` and `pattern` in the second cell to
point at your HDF5 files.

In [ ]:
from caxscripts.scananalysis import DataSet
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
from matplotlib.pyplot import Axes, Line2D
# import siriuspy

In [ ]:
# ── CONFIGURE THESE ──────────────────────────────────────────────────
# workdir = "/home/gabriel.amici/testing_data/2026-03-19"
# workdir = "/home/gabriel.amici/testing_data/2026-03-19"
# workdir = "/mnt/ibira-ids/Carcara-X/Measurements/Raw_Data/2026-08-05/"
# workdir = "/home/gabriel.amici/repos/cax-control/caxsim_scans"
workdir = ("/home/arnaldo.filho/Carcara/Measurements/Analysis/2026/" +
           "2026-08-05/data")
pattern = r"caustic_open_slits_ry_0.4"
# pattern = r"slit"
# ─────────────────────────────────────────────────────────────────────

# caustic_passes = DataSet(workdir, pattern, analysis_mode='fitting')
caustic_passes = DataSet(workdir, pattern, analysis_mode='projection')
caustic_passes

## 1. Explore available data

Each scan pass is a `ScanData` object.  Use `.describe()` to see its
observables (beam properties + metadata keys).

In [ ]:
for scan in caustic_passes.scans:
    scan.describe()
    print()

In [ ]:
# caustic_passes.scan_range = slice()

## 2. Plot observables vs. scanned variable

Multi-component observables (`centroid`, `fwhm`, `intensity`) expand
into separate subplots per component.  Scalar metadata keys (e.g. `ry`,
`cs_rz`) work directly — no registration needed.

In [ ]:
# Plot in window or inline?
%matplotlib qt5
# %matplotlib inline

#### Functions for fitting parabola and hyperbole,
propagating errors of the fitted hyperbole, getting the minimum of the
parabola, and setting axes parameters. 

In [ ]:
def parmin(pol2: tuple) -> tuple:
    """Return the minimum of a polynomial of degree 2."""
    a, b, c = pol2
    delta = b**2 - 4 * a * c
    xmin = -b / (2 * a)
    ymin = -delta / (4 * a)
    return (xmin, ymin)


def hyperbole(z: np.ndarray, a: float, b: float, z0: float) -> np.ndarray:
    """Return the hyperbole y = sqrt(a + b * (x - x0)^2)."""
    return np.sqrt(a + b * (z - z0)**2)


def hyp_std_dev(z: np.ndarray, prm: tuple, stdprm: np.ndarray) -> np.ndarray:
    """Return the standard deviation of the hyperbole.

    y = sqrt(a + b * (x - x0)^2).
    """
    a, b, z0 = prm
    sa, sb, sz0 = stdprm

    dya  = 0.5 / np.sqrt(hyperbole(z, a, b, z0))
    dyb  = (z - z0)**2
    dyz0 = 2 * b * (z - z0)

    return dya * np.sqrt(sa**2 + (sb * dyb)**2 + (sz0 * dyz0)**2)


def plot_fitted_hyperbole(
        axx: Axes,
        zpos: np.ndarray,
        obs: np.ndarray,
        p0: list,
        linex: Line2D,
        labelx: str = "",
        errors: bool = False
        ) -> tuple:
    """Fit a hyperbole to the data and plot it."""
    try:
        prm, cov = curve_fit(hyperbole, zpos, obs, p0=p0)
        stdprm = np.sqrt(np.diag(cov))

        # zfit = np.linspace(zpos.min(), zpos.max(), 100)
        obs_fit = hyperbole(zpos, *prm)
        obs_err = hyp_std_dev(zpos, prm, stdprm)

        labelx = ("fit " + labelx +
                  f", min = ({prm[2]:.2f}, {np.sqrt(prm[0]):.2f})")
        if errors:
            axx.errorbar(zpos, obs_fit, yerr=obs_err, fmt='--',
                         color=linex.get_color(), label=labelx)
        else:
            axx.plot(zpos, obs_fit, '--',
                     color=linex.get_color(), label=labelx)

    except RuntimeError:
        print(f"Fit failed for {labelx}.")
        return None, None

    return prm, cov


def set_axes_prms(ax: Axes, xlabel="", ylabel=""):
    """Set axes parameters."""
    ax.grid(True)
    ax.legend()
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)


### Calculate observables, call fitting functions and plot the curves.

In [ ]:
PIXEL = 0.48  # µm/pixel
fig, (axx, axy)    = plt.subplots(1, 2, figsize=(18, 6))
figc, (axcx, axcy) = plt.subplots(1, 2, figsize=(18, 6))

scs = {}
for ii in range(len(caustic_passes.scans)):
    sc = caustic_passes.scans[ii]
    ry = sc.resolve_observable('mirror.cs_ry')[1][0]

    if round(ry, 4) == 0.4275:
        ry = 0.45

    zpos, (fwhm_x, fwhm_y), obsname = sc.resolve_observable('fwhm')
    zpos, (c_x, c_y), c_obsname = sc.resolve_observable('centroid')
    scs[ry] = {
        'zpos'    : zpos,
        'fwhm_x'  : fwhm_x * PIXEL,
        'fwhm_y'  : fwhm_y * PIXEL,
        'obsname' : obsname,
        'c_x'     : c_x * PIXEL,
        'c_y'     : c_y * PIXEL,
        'c_obsname' : c_obsname
    }

print(f" ry = {scs.keys()}")

for ry in sorted(scs.keys()):

    zpos   = scs[ry]['zpos']
    fwhm_x = scs[ry]['fwhm_x']
    fwhm_y = scs[ry]['fwhm_y']
    c_x    = scs[ry]['c_x']
    c_y    = scs[ry]['c_y']

    # Fitting of a parabola.
    # weightx  = 1 / np.sqrt(fwhm_x)
    # pfx = np.polyfit(zpos, fwhm_x, 2, w=weightx)
    # pnx = np.poly1d(pfx)
    # x_zmin, x_fwhm_min = parmin(pfx)

    # Fitting of a parabola.
    # weighty  = 1 / np.sqrt(fwhm_y)
    # pfy = np.polyfit(zpos, fwhm_y, 2, w=weighty)
    # pny = np.poly1d(pfy)
    # y_zmin, y_fwhm_min = parmin(pfy)

    labelx = f"ry = {ry:.4f} mrad"
    labely = f"ry = {ry:.4f} mrad"
    linex, = axx.plot(zpos, fwhm_x, 'o-', label=labelx)
    liney, = axy.plot(zpos, fwhm_y, 'o-', label=labely)

    axcx.plot(zpos, c_x, 'o-', label=labelx)
    axcy.plot(zpos, c_y, 'o-', label=labely)

    z0 = 0.5 * (zpos[0] + zpos[-1])
    a0x = np.min(fwhm_x)
    a0y = np.min(fwhm_y)
    b0x = 1
    b0y = 1

    # Fitting of a hyperbole for FWHM x.
    ploterrors = True
    prmx, covx = plot_fitted_hyperbole(
        axx, zpos, fwhm_x, p0=[a0x, b0x, z0],
        linex=linex, labelx=labelx, errors=ploterrors
        )

    # Fitting of a hyperbole for FWHM y.
    prmy, covy = plot_fitted_hyperbole(
        axy, zpos, fwhm_y, p0=[a0y, b0y, z0],
        linex=liney, labelx=labely, errors=ploterrors
        )


set_axes_prms(axx, xlabel='z position [mm]', ylabel='FWHM x [µm]')
set_axes_prms(axy, xlabel='z position [mm]', ylabel='FWHM y [µm]')
set_axes_prms(axcx, xlabel='z position [mm]', ylabel='Centroid x [µm]')
set_axes_prms(axcy, xlabel='z position [mm]', ylabel='Centroid y [µm]')

plt.tight_layout()
plt.show()


#### Cells for auxiliary plottings.

In [ ]:
for obs in ['centroid_x', 'centroid_y', 'fwhm_x', 'fwhm_y']:
    caustic_passes.plot_superimposed(obs,
                                     first_item=3,
                                     last_item=17)

In [ ]:
caustic_passes.step_range = slice(1, -1)

In [ ]:
scan0 = caustic_passes.scans[0]
centroid_x_array = scan0.resolve_observable(observable="centroid_x")[1]
scan0.scan_animation(
    observables='centroid', filename='rz_onaxis', y_scale=1000
    )
# scan1.scan_animation(observables='centroid')
# scan0.scan_animation(observables='intensity_peak', filename='tx_showcase_peak')

In [ ]:
# Restrict to a subset of steps
scan.step_range = (2, 18)  # steps 2 … 17
fig, axs = scan.plot_observables(observables=['centroid', 'intensity'])

In [ ]:
# Reset range for subsequent cells
scan.step_range = None

## 3. Per-scan statistics

Extract mean / standard deviation for any observable on a single pass.

In [ ]:
print("Centroid X mean:",     scan.mean_value('centroid_x'))
print("Centroid X std.dev:",  scan.std_deviation('centroid_x'))
print("FWHM X mean:",         scan.mean_value('fwhm_x'))
print("FWHM Y mean:",         scan.mean_value('fwhm_y'))

## 4. Multi-pass statistics

`ScanSet.statistics()` returns mean / median / std across all passes.

In [ ]:
stats = caustic_passes.statistics('centroid_x')
for k, v in stats.items():
    print(f"{k}:  shape={v.shape},  values={v}")

In [ ]:
fig, (axm, axd) = plt.subplots(1, 2, figsize=(15, 5))

idx = stats['xval']
axm.errorbar(idx, stats['mean'], yerr=stats['std_dev'],
             fmt='o-', label='mean')
axm.plot(idx, stats['median'], 's-', label='median', color='red')
axm.set_xlabel('Tx (mm)')
axm.set_ylabel('Centroid X (px)')
axm.set_title('Centroid X — multi-pass statistics')
axm.legend()
axm.grid(True)

axd.plot(idx, stats['std_dev'], 'o-', color='green')
axd.set_xlabel('Tx (mm)')
axd.set_ylabel('Std. Dev. (px)')
axd.set_title('Centroid X — standard deviation')
axd.grid(True)
plt.tight_layout()

## 5. Superimposed traces and correlation

Overlay the same observable from every pass on a single axes.

In [ ]:
fig, ax = caustic_passes.plot_superimposed('intensity_peak')

In [ ]:
corr = caustic_passes.correlation_matrix('centroid_x')
print("Pairwise correlation matrix (centroid_x):")
print(corr)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='viridis', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xlabel('Scan index')
ax.set_ylabel('Scan index')
ax.set_title('centroid_x correlation')
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr[i, j]:.2f}", ha='center', va='center',
                color='w' if abs(corr[i, j]) > 0.5 else 'k')
plt.tight_layout()

## 6. Centroid-vs-motor drift

Look at how a motor position and the centroid X change across passes.

In [ ]:
fig, axs = caustic_passes.centroid_delta_plot('mirror.tx',
                                          step_start=0, step_end=-1)

## 7. Centroid linear fit

Fit a line to the centroid-vs-Tx trace and estimate a rotation angle.

In [ ]:
pixsize = 0.48  # um/px

scan = caustic_passes.scans[0]
xv, (cx, cy) = scan.resolve_observable('centroid')
xv = xv * 1000  # mm -> um  (adjust units as needed)

idx = xv[1:-1]
ax, bx = np.polyfit(idx, cx[1:-1] * pixsize, 1)
ay, by = np.polyfit(idx, cy[1:-1] * pixsize, 1)

fig, (axx, axy) = plt.subplots(1, 2, figsize=(14, 5))

axx.plot(xv, cx * pixsize, 'o-', label='Centroid X')
axx.plot(idx, ax * idx + bx, 'r--', label=f'fit (slope={ax:.4f})')
axx.set_xlabel('Tx (um)')
axx.set_ylabel('Centroid X (um)')
axx.set_title('Centroid X vs Tx')
axx.legend()
axx.grid(True)

axy.plot(xv, cy * pixsize, 'o-', label='Centroid Y')
axy.plot(idx, ay * idx + by, 'r--', label=f'fit (slope={ay:.4f})')
axy.set_xlabel('Tx (um)')
axy.set_ylabel('Centroid Y (um)')
axy.set_title('Centroid Y vs Tx')
axy.legend()
axy.grid(True)

phi = np.arctan(ay / 1000)  # approximate rotation
print(f"Rotation angle (from Y slope): {phi * 1e3:.3f} urad")
plt.tight_layout()

## 8. Beam image + centroid animation

Animate all steps for a single pass: beam image on top, centroid / fwhm
trace below with a marker following the current step.

In [ ]:
scan.observables = ['centroid']
anim = scan.scan_animation(filename=None, fps=2)  # inline

In [ ]:
# Save to file instead
# scan.scan_animation(filename='tx_pass00.gif', fps=2, save_fmt='gif')

## 9. Inspect a single step

Look at a specific step's image, beam properties, and metadata.

In [ ]:
step = scan.steps[5]  # sixth step
print(step)
print()
print("Beam properties:")
for k, v in step.beam_properties.items():
    if k != 'intensity':
        print(f"  {k:>20s} = {v}")
print(f"  {'intensity':>20s} = {step.beam_properties['intensity']}")

In [ ]:
fig, ax = step.plot_image()
ax.set_title(f"Step {step.step_index} — centroid {step.beam_properties['centroid']}")

## 10. Compare DVF A1 vs B1 images

When both detectors are available, view them side by side for any step.

In [ ]:
step = scan.steps[0]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

im1 = ax1.imshow(step.image, cmap='viridis')
plt.colorbar(im1, ax=ax1)
ax1.set_title('DVF B1 (primary)')

if step.image_secondary is not None:
    im2 = ax2.imshow(step.image_secondary, cmap='viridis')
    plt.colorbar(im2, ax=ax2)
    ax2.set_title('DVF A1 (secondary)')
else:
    ax2.text(0.5, 0.5, 'No secondary image',
             ha='center', va='center', transform=ax2.transAxes)

plt.tight_layout()

In [ ]:
print("Metadata keys for this step:")
for k, v in sorted(step.metadata.items()):
    print(f"  {k:30s} = {v}")